# 🛍️ Etsy Digital Products Store — AI Agent System

**Autonomous multi-agent pipeline**: Scrape competitors → Analyze market gaps → Generate products → Optimize listings

Powered by **Gemini API** for all AI deliberation.

---

## 🔑 Setup

Before running, add your Gemini API key:
1. Click the **🔑 Key** icon in the left sidebar
2. Click **+ Add new secret**
3. Name: `GEMINI_API_KEY`, Value: your API key
4. Toggle the **Notebook access** switch ON

Get a free API key at [aistudio.google.com](https://aistudio.google.com/apikey)

## 1️⃣ Install Dependencies

In [ ]:
# Clone the repo and install all dependencies
!git clone https://github.com/28AXE/etsy-store-full-auto.git 2>/dev/null || echo 'Repo already cloned'
import os
os.chdir('/content/etsy-store-full-auto')

# Install Python dependencies
!pip install -q playwright beautifulsoup4 fake-useragent pandas httpx tenacity reportlab google-genai

# Install Playwright Chromium browser
!playwright install chromium
!playwright install-deps chromium

# Make local modules importable
import sys
if '/content/etsy-store-full-auto' not in sys.path:
    sys.path.insert(0, '/content/etsy-store-full-auto')

print('\n✅ All dependencies installed!')

## 2️⃣ Configure API Key & Verify Connection

In [ ]:
import gemini_client

# Quick test — verify Gemini API key works
try:
    result = gemini_client.generate(
        "Say 'Gemini connected!' in exactly 2 words.",
        max_tokens=20
    )
    print(f'✅ Gemini API connected: {result.strip()}')
except Exception as e:
    print(f'❌ API key error: {e}')
    print('\n👉 Add your GEMINI_API_KEY in the Secrets panel (🔑 sidebar)')

---

## 3️⃣ 🕵️ Scout Agent — Scrape Competitor Shop

Scrape an Etsy shop to extract listing data (titles, prices, sales counts).

In [ ]:
import json
from pathlib import Path
from scout.scraper import EtsyScraper

# ========== CONFIGURE ==========
SHOP_NAME = "PlannerSquad"  # Change to any Etsy shop name
OUTPUT_FILE = "data/shop.json"
# ================================

Path("data").mkdir(exist_ok=True)

async def scrape():
    async with EtsyScraper(rate_limit_seconds=3) as scraper:
        shop_data = await scraper.fetch_shop(SHOP_NAME)
        print(f"🕵️ Scraped {shop_data['listing_count']} listings from {SHOP_NAME}")
        print(f"   URL: {shop_data['url']}")

        with open(OUTPUT_FILE, 'w') as f:
            json.dump(shop_data, f, indent=2)
        print(f"\n💾 Saved to {OUTPUT_FILE}")
        return shop_data

shop_data = await scrape()

---

## 4️⃣ 📊 Analyst Agent — Analyze Market Data

Load scraped data (or sample data) and find bestsellers, top tags, and market gaps.

> **Note**: If you don't have scraped data, this cell creates sample data for demonstration.

In [ ]:
import pandas as pd
from analyst.analyzer import MarketAnalyzer

# Create sample data if no scraped data exists
Path("data").mkdir(exist_ok=True)
sample_file = "data/sample_listings.csv"

if not Path(sample_file).exists():
    sample = pd.DataFrame({
        "title": [
            "Digital Daily Planner 2025 Printable",
            "Budget Tracker Spreadsheet Template",
            "Meal Planning Printable Weekly Organizer",
            "Gratitude Journal Digital Download",
            "Fitness Tracker Workout Log Printable",
            "Wedding Planning Checklist Template",
            "Student Planner Academic Year Digital",
            "Habit Tracker Monthly Printable Minimal",
            "Recipe Organizer Digital Cookbook Template",
            "Goal Setting Worksheet New Year Planner",
        ],
        "price": [8.99, 5.49, 7.99, 4.99, 6.49, 12.99, 9.99, 3.99, 8.49, 6.99],
        "sales_count": [245, 189, 156, 312, 98, 67, 201, 423, 134, 88],
        "tags": [
            ["planner", "digital planner", "productivity", "printable"],
            ["budget", "finance", "spreadsheet", "tracker"],
            ["meal plan", "cooking", "organizer", "weekly"],
            ["gratitude", "journal", "self care", "mental health"],
            ["fitness", "workout", "tracker", "health"],
            ["wedding", "checklist", "planning", "bride"],
            ["student", "planner", "academic", "school"],
            ["habit tracker", "minimal", "monthly", "printable"],
            ["recipe", "cookbook", "organizer", "cooking"],
            ["goal setting", "new year", "planner", "productivity"],
        ]
    })
    sample.to_csv(sample_file, index=False)
    print("📝 Created sample data for demonstration")

# Run analysis
analyzer = MarketAnalyzer("data")
analyzer.load_listings("sample_listings.csv")

print("\n" + "="*50)
print("📊 MARKET ANALYSIS RESULTS")
print("="*50)

print(f"\n📦 Total listings: {len(analyzer.df)}")
print(f"💰 Avg price: ${analyzer.df['price'].mean():.2f}")
print(f"📈 Avg sales: {analyzer.df['sales_count'].mean():.0f}")

print("\n🏆 Best Sellers (50+ sales):")
bestsellers = analyzer.find_best_sellers(min_sales=50)
if len(bestsellers) > 0:
    for _, row in bestsellers.iterrows():
        print(f"  • {row['title'][:50]}... — ${row['price']} ({row['sales_count']} sales)")

print("\n🏷️ Top Tags:")
for tag, count in analyzer.extract_top_tags(10):
    print(f"  • {tag}: {count} listings")

print("\n🔍 Market Gaps (high demand, low competition):")
gaps = analyzer.identify_gaps()
for tag, data in sorted(gaps.items(), key=lambda x: x[1]['opportunity_score'], reverse=True)[:5]:
    print(f"  • {tag}: opportunity score {data['opportunity_score']:.2f} (avg {data['avg_sales']:.0f} sales, {data['frequency']} listings)")

### 📊 AI-Powered Deep Analysis (Gemini)

Send the market data to Gemini for strategic insights and niche recommendations.

In [ ]:
# Run Gemini-powered analysis
data_summary = analyzer.prepare_data_for_council()
print("🤖 Sending data to Gemini for analysis...\n")

ai_analysis = analyzer.run_council_analysis(data_summary)

# Pretty-print the results
import json
print(json.dumps(ai_analysis, indent=2))

---

## 5️⃣ 🎨 Creator Agent — Generate Product Content & PDF

Use Gemini to generate Etsy-optimized titles, descriptions, tags, and a printable PDF.

In [ ]:
from creator.content_generator import ContentGenerator
from creator.pdf_generator import PDFGenerator

# ========== CONFIGURE ==========
PRODUCT_TYPE = "Daily Planner"     # Change to your product
TARGET_NICHE = "productivity"      # Target niche
OUTPUT_DIR = "output"              # Output directory for PDFs
# ================================

gen = ContentGenerator()
pdf = PDFGenerator(OUTPUT_DIR)

print("🎨 Generating product content with Gemini...\n")

# Generate title
title = gen.generate_listing_title(
    PRODUCT_TYPE,
    ["digital", "printable", "instant download"],
    TARGET_NICHE
)
print(f"📝 Title: {title.strip()}")

# Generate tags
tags = gen.generate_tags(PRODUCT_TYPE, "productivity enthusiasts")
print(f"\n🏷️ Tags ({len(tags)}): {tags}")

# Generate description
description = gen.generate_description(
    PRODUCT_TYPE,
    ["Instant digital download", "A4 & US Letter sizes", "Undated — use any time",
     "Minimalist clean design", "Print at home or professional print"],
    "Daily planning and goal tracking for busy professionals"
)
print(f"\n📄 Description:\n{description.strip()[:500]}...")

In [ ]:
# Generate the actual PDF product
sections = [
    {
        "title": "Today's Priorities",
        "items": ["Top 3 goals for today", "Most important task", "Deadlines & appointments"]
    },
    {
        "title": "Schedule",
        "items": ["Morning block (6am-12pm)", "Afternoon block (12pm-5pm)", "Evening block (5pm-9pm)"]
    },
    {
        "title": "Notes & Reflections",
        "items": ["What went well today?", "What could improve?", "Tomorrow's top priority"]
    },
]

safe_title = title.strip()[:30].replace(' ', '_').replace('/', '-')
filename = f"{safe_title}.pdf"
filepath = pdf.create_planner(title.strip(), sections, filename)

print(f"\n✅ PDF created: {filepath}")
print(f"   File size: {Path(filepath).stat().st_size:,} bytes")

# Download link in Colab
try:
    from google.colab import files
    files.download(filepath)
    print("\n📥 Download started!")
except ImportError:
    print(f"\n📂 File saved at: {filepath}")

---

## 6️⃣ 📈 Optimizer Agent — Track & Optimize Performance

Log listing metrics and get AI-powered optimization recommendations.

In [ ]:
from optimizer.analytics import PerformanceTracker

tracker = PerformanceTracker("data/analytics")

# Simulate logging metrics over time
sample_metrics = [
    {"views": 150, "favorites": 12, "sales": 3, "revenue": 26.97},
    {"views": 180, "favorites": 15, "sales": 5, "revenue": 44.95},
    {"views": 120, "favorites": 8,  "sales": 2, "revenue": 17.98},
    {"views": 90,  "favorites": 5,  "sales": 1, "revenue": 8.99},
]

for metrics in sample_metrics:
    tracker.log_listing("planner-001", metrics)

# Check trends
trends = tracker.get_listing_trend("planner-001")
print("📈 Performance Trends for planner-001:")
for metric, change in trends.items():
    emoji = "📈" if change > 0 else "📉"
    print(f"  {emoji} {metric}: {change:+.1f}%")

# Get AI optimization recommendations
print("\n🤖 Getting Gemini optimization recommendations...\n")

listings_data = [
    {"listing_id": "planner-001", "title": title.strip(), "trends": trends,
     "current_price": 8.99, "total_sales": 11, "tags": tags[:5]}
]

recommendations = tracker.run_council_optimization(listings_data)
print(json.dumps(recommendations, indent=2))

---

## 7️⃣ 🚀 Full Pipeline — End-to-End Automation

Run the complete pipeline in one cell: Analyze → Generate → Optimize

In [ ]:
import json
from pathlib import Path
from analyst.analyzer import MarketAnalyzer
from creator.content_generator import ContentGenerator
from creator.pdf_generator import PDFGenerator

# ========== CONFIGURE ==========
NICHES = ["Daily Planner", "Budget Tracker", "Meal Planner"]
OUTPUT_DIR = "output/batch"
# ================================

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
gen = ContentGenerator()
pdf = PDFGenerator(OUTPUT_DIR)

print("🚀 FULL PIPELINE — Generating products for all niches")
print("=" * 55)

generated_products = []

for i, niche in enumerate(NICHES, 1):
    print(f"\n--- [{i}/{len(NICHES)}] {niche} ---")

    # Generate content
    title = gen.generate_listing_title(niche, ["digital", "printable"], "seo")
    tags = gen.generate_tags(niche, "digital product buyers")

    # Create PDF
    sections = [
        {"title": "Getting Started", "items": ["Download files", "Print at home or professional print"]},
        {"title": "Features", "items": ["Instant download", "High quality PDF", "Reusable"]},
    ]
    safe_name = niche.replace(' ', '_')
    filepath = pdf.create_planner(title.strip()[:50], sections, f"{safe_name}.pdf")

    product = {
        "niche": niche,
        "title": title.strip(),
        "tags": tags,
        "pdf_path": filepath,
    }
    generated_products.append(product)

    print(f"  ✅ Title: {title.strip()[:70]}...")
    print(f"  ✅ Tags: {len(tags)} generated")
    print(f"  ✅ PDF: {filepath}")

# Summary
print(f"\n{'=' * 55}")
print(f"✅ Generated {len(generated_products)} products!")
print(f"📂 Output directory: {OUTPUT_DIR}")

# Save manifest
manifest_path = f"{OUTPUT_DIR}/manifest.json"
with open(manifest_path, 'w') as f:
    json.dump(generated_products, f, indent=2)
print(f"📋 Manifest saved: {manifest_path}")

---

## 📝 Configuration

Edit the niche targets below and re-run the pipeline cells above.

In [ ]:
# View current niche configuration
with open('config/niches.json') as f:
    niches = json.load(f)

print("📋 Configured Target Niches:")
for niche in niches['target_niches']:
    print(f"\n  🎯 {niche['name']}")
    print(f"     Keywords: {', '.join(niche['keywords'])}")
    print(f"     Price range: ${niche['price_range'][0]} - ${niche['price_range'][1]}")
    print(f"     Competition: {niche['competition']}")